# New Structure Slot Planner Tester

Use this notebook to test the new parallel planner in `src/historyAssistant/planning/new_structure`.

This does **not** use the old Fast Planner or old full LLM planner. It tests:

```text
question -> IntentSlots -> CanonicalPlan -> Validator
```

Edit the question/source settings below and run the cells.

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)

import pandas as pd
from IPython.display import display, JSON, Markdown

from historyAssistant.config import load_source_configs
from historyAssistant.planning.new_structure import try_build_slot_plan

print('Project root:', ROOT)

c:\Users\S_YZH\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: c:\Users\S_YZH\OneDrive - Televic Group NV\Desktop\HistoryAssistant


## 1. Choose One Question

Edit these variables and run the next cells.

In [2]:
# QUESTION = 'Compare May 2024 and June 2024: which month had more calls by ward?'
QUESTION = 'which ward had the most calls last week?'

# Choose one: calls, locations, workflows
SOURCE_ID = 'calls'

SOURCE_TABLE = {
    'calls': 'historyCalls',
    'locations': 'historyLocations',
    'workflows': 'historyWorkflows',
}[SOURCE_ID]

# Optional hints you would normally get from classification/vocabulary normalization.
# Leave empty to test source-default metric inference.
DETECTED_ENTITIES = {
    'comparison_terms': ['compare'],
    'dimensions': ['ward_name'],
}

REQUEST_TYPE = 'qa'
SCENARIO = 'aggregation'
CHART_SUPPRESSED = False
PREFERRED_CHART_TYPE = None

print(QUESTION)

which ward had the most calls last week?


In [3]:
bundle = load_source_configs(SOURCE_ID)
result = try_build_slot_plan(
    question=QUESTION,
    normalized_question=QUESTION,
    source_id=SOURCE_ID,
    target_table=SOURCE_TABLE,
    detected_entities=DETECTED_ENTITIES,
    request_type=REQUEST_TYPE,
    chart_suppressed=CHART_SUPPRESSED,
    preferred_chart_type=PREFERRED_CHART_TYPE,
    bundle=bundle,
    scenario=SCENARIO,
)

debug = result.to_debug_dict()
display(JSON(debug, expanded=False))

c:\Users\S_YZH\OneDrive - Televic Group NV\Desktop\HistoryAssistant\src\historyAssistant\config\dynamic_dimension_loader.py:42: ElasticsearchWarning: Elasticsearch built-in security features are not enabled. Without authentication, your cluster could be accessible to anyone. See https://www.elastic.co/guide/en/elasticsearch/reference/7.17/security-minimal-setup.html to enable security.
  result = es.search(


<IPython.core.display.JSON object>

## 2. Compact Plan View

In [4]:
plan = result.canonical_plan or {}
validation = result.validation or {}
summary = {
    'used': result.used,
    'reason': result.reason,
    'validation_valid': validation.get('valid'),
    'request_type': plan.get('request_type'),
    'metric': plan.get('metric'),
    'aggregation': plan.get('aggregation'),
    'group_by': plan.get('group_by'),
    'filters': plan.get('filters'),
    'date_rule': plan.get('date_rule'),
    'date_range': plan.get('date_range'),
    'compare_to': plan.get('compare_to'),
    'chart_spec': plan.get('chart_spec'),
    'errors': validation.get('errors'),
    'blocking_errors': validation.get('blocking_errors'),
    'repairs': validation.get('repairs'),
}
display(JSON(summary, expanded=True))

<IPython.core.display.JSON object>

## 3. Batch Questions

Edit `TEST_CASES` to test many questions at once. The `entities` field is optional but useful while this new structure is not wired into classification.

In [5]:
TEST_CASES = [
    {
        'question': 'Compare May 2024 and June 2024: which month had more calls by ward?',
        'source_id': 'calls',
        'table': 'historyCalls',
        'scenario': 'aggregation',
        'entities': {'comparison_terms': ['compare'], 'dimensions': ['ward_name']},
    },
    {
        'question': 'Response time by shift in January vs February in 2024',
        'source_id': 'calls',
        'table': 'historyCalls',
        'scenario': 'aggregation',
        'entities': {'metrics': ['avg_response_time'], 'dimensions': ['shift'], 'comparison_terms': ['compare']},
    },
    {
        'question': 'Movement events by area in January vs February in 2024',
        'source_id': 'locations',
        'table': 'historyLocations',
        'scenario': 'aggregation',
        'entities': {'metrics': ['movement_event_count'], 'dimensions': ['area'], 'comparison_terms': ['compare']},
    },
    {
        'question': 'Compare movement activity between patient caba69daba2592b14e91e749f2c6c3f6a7c4b0b9d0a03421d357cdbbcd070eeb and 59104929c3db63ef49fc86f881fa9ce4623e769ec2ee5118ed87f0866cdbe5af on 2024-02-01.',
        'source_id': 'locations',
        'table': 'historyLocations',
        'scenario': 'filtering_ranking',
        'entities': {'dimensions': ['personFirstName'], 'comparison_terms': ['compare']},
    },
]

In [6]:
rows = []
for case in TEST_CASES:
    b = load_source_configs(case['source_id'])
    r = try_build_slot_plan(
        question=case['question'],
        normalized_question=case.get('normalized_question', case['question']),
        source_id=case['source_id'],
        target_table=case['table'],
        detected_entities=case.get('entities', {}),
        request_type=case.get('request_type', 'qa'),
        chart_suppressed=case.get('chart_suppressed', False),
        preferred_chart_type=case.get('preferred_chart_type'),
        bundle=b,
        scenario=case.get('scenario', 'aggregation'),
    )
    p = r.canonical_plan or {}
    v = r.validation or {}
    rows.append({
        'question': case['question'],
        'source': case['source_id'],
        'used': r.used,
        'reason': r.reason,
        'valid': v.get('valid'),
        'request_type': p.get('request_type'),
        'metric': p.get('metric'),
        'group_by': p.get('group_by'),
        'date_range': p.get('date_range'),
        'compare_to': p.get('compare_to'),
        'chart': bool(p.get('chart_spec')),
        'errors': v.get('errors'),
        'blocking_errors': v.get('blocking_errors'),
    })

df = pd.DataFrame(rows)
display(df)

,question,source,used,reason,valid,request_type,metric,group_by,date_range,compare_to,chart,errors,blocking_errors
0,Compare May 2024 and June 2024: which month ha...,calls,True,deterministic_period_comparison:total_calls,True,chart,total_calls,[ward_name],"{'start_date': '2024-05-01', 'end_date': '2024...","{'date_rule': 'custom_range', 'date_range': {'...",True,[],[]
1,Response time by shift in January vs February ...,calls,True,deterministic_period_comparison:avg_response_time,True,chart,avg_response_time,[shift],"{'start_date': '2024-01-01', 'end_date': '2024...","{'date_rule': 'custom_range', 'date_range': {'...",True,[],[]
2,Movement events by area in January vs February...,locations,True,deterministic_period_comparison:movement_event...,True,chart,movement_event_count,[area],"{'start_date': '2024-01-01', 'end_date': '2024...","{'date_rule': 'custom_range', 'date_range': {'...",True,[],[]
3,Compare movement activity between patient caba...,locations,True,deterministic_entity_comparison:movement_event...,True,chart,movement_event_count,[personFirstName],"{'start_date': '2024-02-01', 'end_date': '2024...",None,True,[],[]


## 4. Optional: Execute A Valid Plan

Run this only if Elasticsearch is available. It executes the plan from section 1.

In [7]:
RUN_EXECUTION = False

if RUN_EXECUTION and result.used and result.validation and result.validation.get('valid'):
    from historyAssistant.execution.executor import open_es_connection, execute_canonical_plan
    from historyAssistant.execution.chart_renderer import render_chart
    es = open_es_connection(config_bundle=bundle)
    execution_result = execute_canonical_plan(result.canonical_plan, es, config_bundle=bundle)
    display(JSON({k: v for k, v in execution_result.items() if k not in {'dataframe', 'query', 'chart_payload'}}, expanded=False))
    display(execution_result.get('dataframe'))
    fig = render_chart(execution_result.get('chart_payload'))
    if fig:
        fig.show()
else:
    print('Set RUN_EXECUTION = True to execute the current valid plan.')

Set RUN_EXECUTION = True to execute the current valid plan.
